# YOLO Training with Proper Train/Validation Split
This notebook creates a reproducible split from `inspection.yolov12`, trains a YOLO model on the split dataset, and validates the resulting model.


In [1]:
from pathlib import Path
import random
import shutil

# Paths
src_dir = Path("inspection.yolov12/train")
dst_dir = Path("inspection_split")

# Split configuration
seed = 42
val_ratio = 0.20

# Create split directories
train_images = dst_dir / "train" / "images"
train_labels = dst_dir / "train" / "labels"
val_images = dst_dir / "val" / "images"
val_labels = dst_dir / "val" / "labels"
for path in [train_images, train_labels, val_images, val_labels]:
    path.mkdir(parents=True, exist_ok=True)

# Collect all image files
images = sorted(src_dir.glob("images/*.jpg"))
labels = sorted(src_dir.glob("labels/*.txt"))
assert len(images) == len(labels), "Mismatch between images and labels"

random.seed(seed)
shuffled = images.copy()
random.shuffle(shuffled)

val_count = max(1, int(len(shuffled) * val_ratio))
train_count = len(shuffled) - val_count
train_files = shuffled[:train_count]
val_files = shuffled[train_count:]

for src_image in train_files:
    label_file = src_dir / "labels" / (src_image.stem + ".txt")
    shutil.copy2(src_image, train_images / src_image.name)
    shutil.copy2(label_file, train_labels / label_file.name)

for src_image in val_files:
    label_file = src_dir / "labels" / (src_image.stem + ".txt")
    shutil.copy2(src_image, val_images / src_image.name)
    shutil.copy2(label_file, val_labels / label_file.name)

print(f"Total images: {len(images)}")
print(f"Train images: {len(train_files)}")
print(f"Val images: {len(val_files)}")


Total images: 96
Train images: 77
Val images: 19


In [2]:
import yaml

data = {
    "path": "inspection_split",
    "train": "train/images",
    "val": "val/images",
    "nc": 24,
    "names": [
        "roughness_1","roughness_10","roughness_11","roughness_12","roughness_13","roughness_14",
        "roughness_15","roughness_16","roughness_17","roughness_18","roughness_19","roughness_2",
        "roughness_20","roughness_21","roughness_22","roughness_23","roughness_24",
        "roughness_3","roughness_4","roughness_5","roughness_6","roughness_7","roughness_8","roughness_9"
    ]
}
with open(dst_dir / "data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Wrote", dst_dir / "data.yaml")


Wrote inspection_split/data.yaml


In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("Loaded model", model)


Loaded model YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, tr

In [5]:
model.train(
    data=str(dst_dir / "data.yaml"),
    epochs=5,
    imgsz=640,
    batch=8,
    device="cpu",
    name="steel_detector_split"
)


New https://pypi.org/project/ultralytics/8.4.43 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.38 🚀 Python-3.12.3 torch-2.11.0+cu130 CPU (Intel Core i7-8550U 1.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=inspection_split/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=st

KeyboardInterrupt: 

In [ ]:
metrics = model.val(data=str(dst_dir / "data.yaml"))
print(metrics)


In [ ]:
from PIL import Image
from IPython.display import display

sample_img = list((dst_dir / "val" / "images").glob("*.jpg"))[0]
results = model(str(sample_img), show=False)
result_img = results[0].plot()
display(Image.fromarray(result_img))
print("Sample image:", sample_img)
